# Feedforward → Direct Mesh

Demonstrates meshing directly from a MapAnything pointcloud without training a Gaussian splat. The reconstruction is loaded from the zarr cache written by `feedforward_methods.ipynb` — run that notebook first.

**Pipeline:** zarr cache → `FeedforwardResult.load_zarr()` → `fuse_tsdf()` → `clean_repair_mesh()` → mesh

`fuse_tsdf` takes plain arrays, never a result object: the caller composes depth, RGB, camera-to-world poses and intrinsics, because only the caller knows which resolution grid it is on.

Gaussian-splat training lives in the `splats` stage (`collab_splats/splats/`).

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from pathlib import Path

import numpy as np
import open3d as o3d

from collab_splats.geometry.transforms import invert_poses
from collab_splats.mesh import clean_repair_mesh, fuse_tsdf
from collab_splats.pointcloud.feedforward.base import FeedforwardResult

## §1 — Load cached reconstruction

Read the MapAnything result written by `feedforward_methods.ipynb`. The zarr store lives in `docs/source/.cache/` and is gitignored; run the prior notebook if it is absent.

In [ ]:
%run ../tutorial_config.py

# ── Configuration ─────────────────────────────────────────────────────────────
METHOD = "mapanything"

RECON = OUTPUT_DIR / METHOD

# Load from zarr written by feedforward_methods.ipynb
_recon_cache = RECON / "reconstruction.zarr"
assert _recon_cache.exists(), (
    f"No reconstruction cache at {_recon_cache}. " "Run 02_pointcloud/feedforward_methods first."
)

result = FeedforwardResult.load_zarr(_recon_cache)
print(f"Loaded MapAnything reconstruction: {result.points.shape[0]:,} pts, {result.extrinsics.shape[0]} frames")

## §2 — TSDF meshing

Run TSDF fusion directly on the feedforward depth maps. `voxel_size=0.02` (2 cm) is coarser than the splat-derived path because feedforward depth maps are noisier than splat-rendered depths.

In [ ]:
MESH_DIR = Path("/tmp/feedforward_mesh_demo/mesh")

# Model-resolution arrays straight off the forward pass. images is (N, 3, H, W) float in
# [0, 1]; fuse_tsdf takes (N, H, W, 3) uint8 and camera-to-world poses.
rgbs = np.asarray(result.images).transpose(0, 2, 3, 1)
rgbs = np.ascontiguousarray((np.clip(rgbs, 0.0, 1.0) * 255).round().astype(np.uint8))

mesh_path = fuse_tsdf(
    np.asarray(result.depth),
    rgbs,
    invert_poses(result.extrinsics),
    result.intrinsics,
    MESH_DIR,
    voxel_size=0.005,
    depth_trunc=1.0,
)

# Cleaning is a separate call: fusion decides geometry, cleaning decides what to keep.
# Thresholds are fractions of the mesh's own extent, so there is nothing to tune per scene.
clean_repair_mesh(mesh_path)
print(f"Mesh saved → {mesh_path}")


TSDF integration:   0%|          | 0/80 [00:00<?, ?it/s]


TSDF integration:   1%|▏         | 1/80 [00:00<01:04,  1.22it/s]


TSDF integration:   2%|▎         | 2/80 [00:01<00:36,  2.16it/s]


TSDF integration:   4%|▍         | 3/80 [00:01<00:29,  2.60it/s]


TSDF integration:   5%|▌         | 4/80 [00:01<00:23,  3.20it/s]

...


TSDF integration:  98%|█████████▊| 78/80 [00:41<00:02,  1.22s/it]


TSDF integration:  99%|█████████▉| 79/80 [00:42<00:01,  1.12s/it]


TSDF integration: 100%|██████████| 80/80 [00:43<00:00,  1.03s/it]


TSDF integration: 100%|██████████| 80/80 [00:43<00:00,  1.85it/s]

Mesh saved → /tmp/feedforward_mesh_demo/mesh/mesh.ply


## §3 — Inspect mesh

Load the saved mesh with open3d and report basic statistics.

In [5]:
# Inspect mesh statistics
mesh = o3d.io.read_triangle_mesh(str(mesh_path))
print(f"Vertices: {len(mesh.vertices):,}  Triangles: {len(mesh.triangles):,}")

Vertices: 2,752,669  Triangles: 4,712,371
